# CampusSpot: เช็กข้อมูลรอบ มจธ. บางมด (เวอร์ชัน 2)

**สิ่งที่เปลี่ยนจากเวอร์ชันแรก**
- ใช้ Overture เป็นแหล่งหลัก และใช้ OpenStreetMap เสริม (รอบแรกพบว่า Overture ครบกว่ามาก)
- กฎจัดประเภทใหม่: ตัดห้องสมุด โรงอาหาร ตลาด บาร์ ร้านขนส่งพัสดุ ออกจากหมวดหอพัก, ตัดร้านอินเทอร์เน็ตออกจากร้านกาแฟ, แยกโรงพิมพ์ออกจากร้านปริ้นต์, ตัดร้านล้างรถออกจากร้านซักผ้า
- ดึงร้านซักผ้าและร้านปริ้นต์ที่ตกหล่นไปอยู่หมวดอื่นกลับมาด้วยการค้นชื่อ
- ดึงขอบเขตมหาวิทยาลัยจาก OpenStreetMap แล้วแยกร้านในมหาวิทยาลัยออกอัตโนมัติ
- ตัดรายการซ้ำข้ามแหล่งข้อมูล โดยไม่รวมหอที่ต่างกันแค่เลขอาคาร (เช่น หอ 1 กับหอ 2)
- **ไม่ลบข้อมูลทิ้ง** ทุกจุดยังอยู่ แต่มีคอลัมน์ `flag` บอกสถานะ และคอลัมน์ `reason` บอกเหตุผล
- เพิ่มเครื่องมือเทียบชื่อหอพักกับรายชื่อจากเว็บรวมหอพัก และแบบฟอร์มเดินสำรวจ
- บันทึกผลลง Google Drive ได้ และดาวน์โหลดเป็น zip ไฟล์เดียว

รันบน Google Colab: กด "เรียกใช้ทุกเซลล์" ได้เลย (Earth Engine ปิดไว้ก่อน เปิดได้ที่ `RUN_EE` ในส่วนที่ 8)

In [ ]:
!pip install -q overturemaps geopandas folium shapely requests

In [ ]:
import math, re, os, json, difflib, zipfile, requests
import pandas as pd, geopandas as gpd, folium
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import unary_union, polygonize

LAT, LON = 13.6513, 100.4941      # กลาง มจธ. บางมด
RADIUS_M = 1500
SAVE_TO_DRIVE = False             # เปลี่ยนเป็น True ถ้าอยากบันทึกลง Google Drive อัตโนมัติ
DRIVE_DIR = "/content/drive/MyDrive/campusspot"
OUT = "out"

dlat = RADIUS_M / 111_320
dlon = RADIUS_M / (111_320 * math.cos(math.radians(LAT)))
BBOX = (LON - dlon, LAT - dlat, LON + dlon, LAT + dlat)
CATS = ["dorm_like", "cafe", "food", "laundry", "print_stationery", "convenience"]

def dist_m(lat, lon, lat0=LAT, lon0=LON):
    return math.hypot((lat - lat0) * 111_320, (lon - lon0) * 111_320 * math.cos(math.radians(lat0)))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = DRIVE_DIR + "/out"
os.makedirs(OUT, exist_ok=True)
print("output folder:", OUT)

## 1) ดึงข้อมูลจาก OpenStreetMap (ร้านค้า หอพัก จำนวนอาคาร และขอบเขตมหาวิทยาลัย)

In [ ]:
import time
HEADERS = {"User-Agent": "CampusSpot-KMUTT/0.2 (student project)", "Accept": "application/json"}
MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
def overpass(q, rounds=4):
    """ลองทุกเซิร์ฟเวอร์ วนหลายรอบ รอนานขึ้นเรื่อย ๆ ถ้าเซิร์ฟเวอร์ไม่ว่าง (406/429/502/503/504)"""
    last = None
    for rnd in range(rounds):
        for url in MIRRORS:
            try:
                r = requests.post(url, data={"data": q}, headers=HEADERS, timeout=200)
                if r.status_code == 200:
                    try:
                        data = r.json()
                    except ValueError:
                        last = f"non-JSON from {url}"; print("  ✗", last); continue
                    print("  ✓ ok from", url)
                    return data["elements"]
                last = f"{r.status_code} from {url}"
            except requests.RequestException as e:
                last = f"{type(e).__name__} from {url}"
            print("  ✗", last)
            time.sleep(3)
        wait = 15 * (rnd + 1)
        print(f"รอบ {rnd+1} ไม่สำเร็จ รอ {wait} วินาทีแล้วลองใหม่...")
        time.sleep(wait)
    raise RuntimeError(f"Overpass ทุกเซิร์ฟเวอร์ไม่ว่าง (ล่าสุด: {last}) ลองรันเซลล์นี้ใหม่อีก 10 นาที")

A = f"(around:{RADIUS_M},{LAT},{LON})"
osm_els = overpass(f'''[out:json][timeout:120];
(
  nwr{A}["amenity"~"cafe|restaurant|fast_food|food_court|ice_cream|internet_cafe|dormitory"];
  nwr{A}["shop"];
  nwr{A}["building"~"dormitory|apartments"];
  nwr{A}["tourism"~"guest_house|hostel|apartment"];
  nwr{A}["name"~"หอพัก|อพาร์ท|แมนชั่น|Apartment|Mansion|Residence|Dorm",i];
);
out center tags;''')
print("OSM elements:", len(osm_els))

building_count = int(overpass(f'[out:json][timeout:120];way{A}["building"];out count;')[0]["tags"]["ways"])
print("OSM building footprints:", building_count)

In [ ]:
def build_polys(els):
    polys = []
    for e in els:
        if e["type"] == "way" and "geometry" in e:
            pts = [(p["lon"], p["lat"]) for p in e["geometry"]]
            if len(pts) >= 4:
                polys.append(Polygon(pts))
        elif e["type"] == "relation":
            lines = [LineString([(p["lon"], p["lat"]) for p in m["geometry"]])
                     for m in e.get("members", []) if m.get("role") == "outer" and "geometry" in m]
            polys += list(polygonize(unary_union(lines)))
    return polys

campus_els = overpass(f'''[out:json][timeout:60];
(way(around:800,{LAT},{LON})["amenity"="university"];
 relation(around:800,{LAT},{LON})["amenity"="university"];);
out geom;''')
cands = [p.buffer(0) for p in build_polys(campus_els)]
center = Point(LON, LAT)
campus_poly = max(cands, key=lambda p: (p.contains(center), p.area)) if cands else None
if campus_poly is None:
    print("ไม่พบขอบเขตมหาวิทยาลัยใน OSM: จะใช้แค่ชื่อร้านในการแยกร้านในมหาวิทยาลัย")
else:
    approx_km2 = campus_poly.area * (111.32 ** 2) * math.cos(math.radians(LAT))
    print(f"campus polygon: ~{approx_km2:.2f} ตร.กม., contains center = {campus_poly.contains(center)}")

## 2) ดึงข้อมูลจาก Overture Maps (แหล่งหลัก)

In [ ]:
from overturemaps import core
try:
    ov = core.geodataframe("place", bbox=BBOX)
    print("Overture places in bbox:", len(ov))
except Exception as e:
    print("ดึง Overture ไม่สำเร็จ:", type(e).__name__, e)
    print("จะใช้แค่ OSM ไปก่อน (Overture เป็นแหล่งหลัก ควรรันเซลล์นี้ใหม่ภายหลัง)")
    ov = None


## 3) จัดประเภท

กฎทั้งหมดอยู่ในเซลล์นี้ ถ้าเจอร้านที่จัดผิด ให้แก้ regex ด้านบนแล้วรันเซลล์นี้กับเซลล์ถัดไปใหม่

In [ ]:
DORM_STRONG = r"หอพัก|หอใน|หอชาย|หอหญิง|อพาร์ท|อพาร์ต|แมนชั่น|apartment|mansion|residence|dorm"
DORM_WEAK   = r"เพลส|place|living|เรสซิเดนซ์|คอนโด|condo|^หอ(?!ม|สมุด|ประชุม|การค้า)"
DORM_FP     = r"หอสมุด|library(?!.*condo)|โรงอาหาร|market ?place|fitness|เอ็กเพลส|express|หอมสนิท|บาร์|\bbar\b|หมู่บ้าน|village|ร้านอาหาร|cafe|coffee"
INTERNET    = r"internet|อินเทอร์เน็ต|อินเตอร์เน็ต|เกมส์|game"
LAUNDRY     = r"ซัก|laundr|laundromat|wash ?(&|and)? ?dry|wash express|dwash|dry ?clean"
NOT_LAUNDRY = r"car|ล้างรถ|detailing|pet|สุนัข"
PRINT       = r"ถ่ายเอกสาร|copy|ปริ้น|พริ้น|print|เครื่องเขียน|stationar|stationer|เข้าเล่ม|ทำเล่ม"
PRINT_STUDENT = r"ถ่ายเอกสาร|copy|ปริ้นสี|เข้าเล่ม|ทำเล่ม|digital|เครื่องเขียน|stationar|stationer|postshop|overprint"
PRINT_INDUSTRIAL = r"co\.|ltd|จำกัด|บริษัท|โรงพิมพ์|import|อิมปอร์ต|ป้าย|สกรีน|ฉลาก|สติ๊กเกอร์|packag|label|การพิมพ์|printing"
ON_CAMPUS_NAME = r"kmutt|มจธ|ตึกอธิการ|residencehall"

def has(p, s): return bool(re.search(p, s or "", re.I))

def classify(name, signals):
    s = (signals or "").lower()
    n = name or ""
    if has(INTERNET, n) or "internet" in s: return "other"
    if has(LAUNDRY, n) or "laundr" in s or "dry_clean" in s:
        return "other" if has(NOT_LAUNDRY, n) else "laundry"
    if has(PRINT, n) or "print" in s or "copy" in s or "stationer" in s: return "print_stationery"
    if any(k in s for k in ["coffee", "cafe", "tea_room", "bubble_tea"]): return "cafe"
    if any(k in s for k in ["restaurant", "fast_food", "food_court", "food", "ice_cream", "noodle", "bakery"]): return "food"
    if any(k in s for k in ["convenience", "supermarket"]): return "convenience"
    if has(DORM_STRONG, n) or any(k in s for k in ["dormitor", "apartment", "hostel", "guest_house"]): return "dorm_like"
    if has(DORM_WEAK, n) and not any(k in s for k in ["shop", "store", "restaurant", "cafe", "service"]): return "dorm_like"
    return "other"

def osm_signals(t):
    keys = ["amenity", "shop", "building", "tourism", "cuisine"]
    s = " ".join(f"{k}={t.get(k, '')}" for k in keys if t.get(k))
    if t.get("amenity") == "cafe" or t.get("shop") == "coffee": s += " coffee"
    if t.get("amenity") in ("restaurant", "fast_food", "food_court", "ice_cream"): s += " restaurant"
    if t.get("shop") in ("laundry", "dry_cleaning"): s += " laundry"
    if t.get("shop") in ("copyshop", "stationery"): s += " print stationer"
    if t.get("shop") in ("convenience", "supermarket"): s += " convenience"
    if t.get("amenity") == "internet_cafe": s += " internet"
    if t.get("amenity") == "dormitory" or t.get("building") in ("dormitory", "apartments") \
       or t.get("tourism") in ("guest_house", "hostel", "apartment"): s += " apartment"
    return s

def first(x, key):
    if isinstance(x, dict): return x.get(key)
    return None

def ov_signals(cat, basic=None):
    parts = [str(basic)] if isinstance(basic, str) and basic else []
    if isinstance(cat, dict):
        alt = cat.get("alternate")
        alt = list(alt) if alt is not None else []
        parts += [str(cat.get("primary") or "")] + [str(a) for a in alt]
    return " ".join(parts)

In [ ]:
rows = []
for e in osm_els:
    t = e.get("tags", {})
    lat = e.get("lat") or e.get("center", {}).get("lat")
    lon = e.get("lon") or e.get("center", {}).get("lon")
    if lat is None: continue
    sig = osm_signals(t)
    rows.append({"source": "osm", "name": t.get("name", ""), "signals": sig, "confidence": None,
                 "lat": lat, "lon": lon})
for _, r in (ov.iterrows() if ov is not None else []):
    rows.append({"source": "overture", "name": first(r.get("names"), "primary") or "",
                 "signals": ov_signals(r.get("categories"), r.get("basic_category")),
                 "confidence": r.get("confidence"),
                 "lat": r.geometry.y, "lon": r.geometry.x})
pts = pd.DataFrame(rows)
pts["confidence"] = pd.to_numeric(pts["confidence"], errors="coerce")
pts["dist_m"] = [round(dist_m(a, b)) for a, b in zip(pts.lat, pts.lon)]
pts = pts[pts.dist_m <= RADIUS_M].reset_index(drop=True)
pts["category"] = [classify(n, s) for n, s in zip(pts.name, pts.signals)]
print(pts.groupby(["category", "source"]).size().unstack(fill_value=0).reindex(CATS + ["other"], fill_value=0))

## 4) ติดธง: จับผิด, โรงพิมพ์, ในมหาวิทยาลัย, รายการซ้ำ

In [ ]:
pts["flag"], pts["reason"] = "ok", ""
def setf(mask, flag, reason):
    m = mask & (pts.flag == "ok")
    pts.loc[m, "flag"], pts.loc[m, "reason"] = flag, reason

nm = pts.name.fillna("")
setf((pts.category == "dorm_like") & nm.str.contains(DORM_FP, case=False, regex=True), "false_positive", "ชื่อไม่ใช่หอพัก")
setf((pts.category == "dorm_like") & nm.str.fullmatch(r"\s*[A-Za-z0-9]?\s*"), "needs_check", "ไม่มีชื่อหรือชื่อสั้นเกินไป")
setf((pts.category == "dorm_like") & ~nm.str.contains(DORM_STRONG, case=False, regex=True)
     & ~pts.signals.str.contains("apartment|dormitor|hostel|guest_house", case=False), "needs_check", "มีแค่คำอ่อน เช่น Place")
setf((pts.category == "print_stationery") & nm.str.contains(PRINT_INDUSTRIAL, case=False, regex=True)
     & ~nm.str.contains(PRINT_STUDENT, case=False, regex=True), "industrial_print", "โรงพิมพ์หรือบริษัท ไม่ใช่ร้านสำหรับนักศึกษา")

if campus_poly is not None:
    inside = [campus_poly.contains(Point(b, a)) for a, b in zip(pts.lat, pts.lon)]
    setf(pd.Series(inside), "on_campus", "อยู่ในขอบเขตมหาวิทยาลัย")
setf(nm.str.contains(ON_CAMPUS_NAME, case=False, regex=True) & (pts.dist_m < 600), "on_campus", "ชื่อบอกว่าอยู่ในมหาวิทยาลัย")

ALIAS = {"cosmo": "คอสโม", "thonburisport": "ธนบุรีสปอร์ต", "siri": "สิริ", "alisha": "อลิชา",
         "dhammaraksa": "ธรรมรักษา", "7eleven": "เซเว่น", "lotus": "โลตัส", "amazon": "อเมซอน",
         "kruajeleng": "เจ๊เล้ง"}   # เพิ่มคู่ชื่อไทย-อังกฤษที่เจอได้เรื่อย ๆ
GENERIC = r"^(หอพักหญิง|หอพักชาย|หอพัก|บ้าน|ร้าน|ก๋วยเตี๋ยว|ข้าว|ครัว|คาเฟ่|cafe|coffee|the)+"
def key(s):
    s = re.sub(r"[^a-z0-9ก-๙]", "", str(s).lower())
    for en, th in ALIAS.items():
        s = s.replace(en, th)
    return re.sub(GENERIC, "", s)
def trailing_num(s):
    m = re.search(r"(\d+)\D*$", str(s))
    return m.group(1) if m else None
def similar(ka, kb):
    if min(len(ka), len(kb)) < 4: return False
    if ka in kb or kb in ka: return True
    if len(ka) >= 8 and ka[:8] == kb[:8]: return True
    return difflib.SequenceMatcher(None, ka, kb).ratio() >= 0.8

DUP_RADIUS_M = 60   # ร้านชื่อคล้ายกันในระยะนี้ถือว่าซ้ำ (ร้านเชนสาขาต่างกันมักห่างกว่านี้)
pts["_k"] = pts.name.map(key)
order = pts.sort_values(["source", "confidence"], ascending=[False, False]).index  # overture ก่อน
recs = pts.to_dict("index")
grid = {}   # (category, ช่องกริด) -> รายการจุดที่เก็บไว้แล้ว
cell = lambda lat, lon: (int(lat * 111_320 // DUP_RADIUS_M), int(lon * 108_000 // DUP_RADIUS_M))
for i in order:
    a = recs[i]
    if a["flag"] == "false_positive": continue
    gx, gy = cell(a["lat"], a["lon"])
    dup_of = None
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for j in grid.get((a["category"], gx + dx, gy + dy), []):
                b = recs[j]
                if dist_m(a["lat"], a["lon"], b["lat"], b["lon"]) > DUP_RADIUS_M: continue
                na, nb = trailing_num(a["name"]), trailing_num(b["name"])
                if na and nb and na != nb: continue
                if similar(a["_k"], b["_k"]):
                    dup_of = j; break
            if dup_of is not None: break
        if dup_of is not None: break
    if dup_of is None:
        grid.setdefault((a["category"], gx, gy), []).append(i)
    elif a["flag"] == "ok":
        pts.at[i, "flag"], pts.at[i, "reason"] = "duplicate", f"ซ้ำกับ: {recs[dup_of]['name']}"
pts = pts.drop(columns="_k")
print(pts.flag.value_counts())

In [ ]:
target = pts[pts.category.isin(CATS)]
summary = pd.DataFrame({
    "raw_osm": target[target.source == "osm"].groupby("category").size(),
    "raw_overture": target[target.source == "overture"].groupby("category").size(),
    "usable": target[target.flag == "ok"].groupby("category").size(),
    "on_campus": target[target.flag == "on_campus"].groupby("category").size(),
    "needs_check": target[target.flag == "needs_check"].groupby("category").size(),
}).reindex(CATS).fillna(0).astype(int)
summary

In [ ]:
# ดูรายการที่ถูกติดธง เพื่อตรวจว่ากฎตัดถูกไหม (แก้กฎในเซลล์จัดประเภทได้)
for f in ["false_positive", "industrial_print", "needs_check", "on_campus", "duplicate"]:
    sub = pts[(pts.flag == f) & pts.category.isin(CATS)]
    print(f"\n=== {f} ({len(sub)}) ===")
    print(sub[["category", "source", "name", "reason"]].head(40).to_string(index=False))

In [ ]:
m = folium.Map(location=[LAT, LON], zoom_start=15, tiles=None)
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri", name="แผนที่ถนน").add_to(m)
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri", name="ภาพดาวเทียม").add_to(m)
folium.Circle([LAT, LON], radius=RADIUS_M, color="#888", fill=False).add_to(m)
if campus_poly is not None:
    folium.GeoJson(gpd.GeoSeries([campus_poly]).__geo_interface__,
                   style_function=lambda _: {"color": "#555", "weight": 1, "fillOpacity": 0.08}).add_to(m)
colors = {"dorm_like": "purple", "cafe": "orange", "food": "red", "laundry": "blue",
          "print_stationery": "green", "convenience": "gray"}
for c in CATS:
    layer = folium.FeatureGroup(name=c)
    for _, p in pts[pts.category == c].iterrows():
        ok = p["flag"] == "ok"
        folium.CircleMarker([p["lat"], p["lon"]], radius=5 if ok else 3, weight=1 if ok else 2,
                            color=colors[c] if ok else "#999", fill=ok, fill_opacity=0.8,
                            tooltip=f"{c} | {p['flag']} | {p['name']}").add_to(layer)
    layer.add_to(m)
folium.LayerControl().add_to(m)
m.save(f"{OUT}/map.html")
m  # จุดทึบ = ใช้ได้, จุดเทาโปร่ง = ถูกติดธง (เปิด/ปิดแต่ละประเภทได้ที่มุมขวาบน)

## 5) เทียบชื่อหอพักกับเว็บรวมหอพัก (งานโต๊ะ ไม่ต้องเดิน)

คัดลอกชื่อหอพักที่อยู่ในรัศมีจากเว็บรวมหอพักมาประมาณ 30 ชื่อ วางในตัวแปร `REF_DORMS` (บรรทัดละหนึ่งชื่อ)
ระบบจะจับคู่แบบคลาดเคลื่อนได้ แล้วบอกว่าข้อมูลของเราครอบคลุมกี่เปอร์เซ็นต์

In [ ]:
REF_DORMS = '''
'''.strip().splitlines()

ours = pts[(pts.category == "dorm_like") & pts.flag.isin(["ok", "on_campus", "needs_check"])]
DORM_GENERIC = r"หอพัก|หอ|อพาร์[ทต]เม?[นน]?[ทต]์?|อพาร์ท|อพาร์ต|เม้นท์|เมนท์|เมนต์|แมนชั่น|เพลส|เรสซิเดนซ์|apartment|mansion|place|residence|ใกล้|มจธ|kmutt|บางมด"
def dorm_key(s):
    s = str(s).lower()
    for en, th in ALIAS.items():
        s = s.replace(en, th)
    s = re.sub(DORM_GENERIC, "", s)
    return re.sub(r"[^a-z0-9ก-๙]", "", s)
our_keys = {dorm_key(n): n for n in ours.name if n and len(dorm_key(n)) >= 2}
res = []
for ref in [r.strip() for r in REF_DORMS if r.strip()]:
    k = dorm_key(ref)
    best = max(our_keys, key=lambda x: difflib.SequenceMatcher(None, k, x).ratio(), default=None)
    score = difflib.SequenceMatcher(None, k, best).ratio() if best else 0
    res.append({"reference": ref, "best_match": our_keys.get(best, ""), "score": round(score, 2), "found": score >= 0.7})
dorm_match = pd.DataFrame(res)
if len(dorm_match):
    print(f"พบ {dorm_match.found.sum()} จาก {len(dorm_match)} ชื่อ = {dorm_match.found.mean():.0%}")
    display(dorm_match.sort_values("score"))
    print("คอลัมน์ found เป็นการเดาอัตโนมัติ ชื่อไทยกับอังกฤษ (เช่น กรีนไลฟ์เพลส กับ Green Life Place) จับคู่กันไม่ได้ ให้ไล่ดู best_match ด้วยตาแล้วแก้เองก่อนสรุป")
else:
    print("ยังไม่ได้ใส่ชื่อหอพักใน REF_DORMS")

## 6) เดินสำรวจภาคสนาม

เซลล์แรกสร้างไฟล์ `field_survey_template.csv` ที่มีร้านจากข้อมูลของเราใน 2 โซนตัวอย่างอยู่แล้ว
นำไปเปิดใน Google Sheets บนมือถือ เดินดูแล้วกรอกคอลัมน์ `exists` (yes/no) และเพิ่มแถวใหม่สำหรับร้านที่ไม่มีในข้อมูล (ใส่ `in_data` = no)
อย่าลืมนับเครื่องซักผ้าหยอดเหรียญในหอพักในหมวด laundry ด้วย

In [ ]:
ZONES = {  # ปรับพิกัดและรัศมีตามที่เดินจริง
    "A_front_pracha_uthit": (13.6505, 100.4960, 250),
    "B_pracha_uthit_45":    (13.6460, 100.4990, 250),
}
FIELD_CATS = ["laundry", "print_stationery", "food", "cafe", "dorm_like"]
tpl = []
for z, (zl, zo, zr) in ZONES.items():
    sub = pts[(pts.flag == "ok") & pts.category.isin(FIELD_CATS)]
    sub = sub[[dist_m(a, b, zl, zo) <= zr for a, b in zip(sub.lat, sub.lon)]]
    for _, p in sub.iterrows():
        tpl.append({"zone": z, "category": p["category"], "name": p["name"], "lat": p["lat"], "lon": p["lon"],
                    "in_data": "yes", "exists": "", "note": ""})
pd.DataFrame(tpl).to_csv(f"{OUT}/field_survey_template.csv", index=False, encoding="utf-8-sig")
print(f"สร้างแบบฟอร์ม {len(tpl)} แถว ใน {len(ZONES)} โซน -> {OUT}/field_survey_template.csv")

In [ ]:
# หลังกรอกเสร็จ อัปโหลดไฟล์กลับมาที่ Colab แล้วรันเซลล์นี้
FILLED = f"{OUT}/field_survey_filled.csv"
if os.path.exists(FILLED):
    f = pd.read_csv(FILLED)
    f["exists"] = f["exists"].astype(str).str.lower().str.strip()
    real = f[f.exists == "yes"]
    rec = real.groupby("category").apply(lambda d: (d.in_data == "yes").mean()).rename("recall")
    cnt = real.groupby("category").size().rename("real_count")
    stale = f[(f.in_data == "yes") & (f.exists == "no")].groupby("category").size().rename("closed_or_wrong")
    field_result = pd.concat([cnt, rec, stale], axis=1).fillna(0)
    field_result["verdict"] = pd.cut(field_result.recall, [-0.01, 0.3, 0.6, 1.01], labels=["ไม่พอ", "ต้องเก็บเสริม", "ใช้ได้"])
    display(field_result)
else:
    print("ยังไม่มีไฟล์", FILLED)

## 7) บันทึกและดาวน์โหลดผลทั้งหมด

In [ ]:
pts.sort_values(["category", "flag", "dist_m"]).to_csv(f"{OUT}/points_cleaned.csv", index=False, encoding="utf-8-sig")
summary.to_csv(f"{OUT}/summary.csv", encoding="utf-8-sig")
meta = {"center": [LAT, LON], "radius_m": RADIUS_M,
        "osm_building_count": building_count if "building_count" in globals() else None,
        "campus_polygon_found": campus_poly is not None}
if "dorm_match" in globals() and len(dorm_match):
    dorm_match.to_csv(f"{OUT}/dorm_name_check.csv", index=False, encoding="utf-8-sig")
    meta["dorm_name_recall"] = float(dorm_match.found.mean())
json.dump(meta, open(f"{OUT}/meta.json", "w"), ensure_ascii=False, indent=2)

zip_path = "campusspot_out_v2.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for fn in os.listdir(OUT):
        z.write(os.path.join(OUT, fn), fn)
print("saved:", sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("ไม่ได้รันบน Colab: ไฟล์อยู่ที่", zip_path)

## 8) ตัวเลือก: เช็กการสร้างอาคารใหม่จากดาวเทียม (Google Earth Engine)

ต้องสมัคร Earth Engine และใส่ชื่อโปรเจกต์ Cloud ของคุณ

In [ ]:
RUN_EE = False   # เปลี่ยนเป็น True เมื่อพร้อมล็อกอิน Earth Engine
EE_PROJECT = "YOUR-CLOUD-PROJECT"   # ใส่ชื่อโปรเจกต์ Cloud ของคุณ

if not RUN_EE:
    print("ข้ามส่วน Earth Engine (RUN_EE = False)")
else:
    import ee
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

    aoi = ee.Geometry.Point([LON, LAT]).buffer(RADIUS_M)
    def built(y):
        return (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(aoi)
                .filterDate(f"{y}-01-01", f"{y}-12-31").select("built").mean().clip(aoi))
    diff = built(2025).subtract(built(2019))
    area_m2 = diff.gt(0.3).multiply(ee.Image.pixelArea()).reduceRegion(ee.Reducer.sum(), aoi, 10).getInfo()["built"]
    print(f"พื้นที่ที่กลายเป็นสิ่งปลูกสร้างใหม่ 2019-2025: ประมาณ {area_m2/10_000:.1f} เฮกตาร์")
    print(diff.getThumbURL({"min": -0.5, "max": 0.5, "palette": ["2166ac", "f7f7f7", "b2182b"], "dimensions": 600, "region": aoi}))


## เกณฑ์ตัดสินใจ

- ความครบจากการเดินสำรวจหรือการเทียบชื่อหอพัก **60% ขึ้นไป**: ใช้ข้อมูลเปิดเป็นหลักได้
- **30–60%**: ใช้ได้ แต่กันเวลา 3–4 วันเก็บข้อมูลเสริมในพื้นที่เดโม
- **ต่ำกว่า 30%**: ข้อมูลเปิดใช้ไม่ได้ในหมวดนั้น ต้องเก็บเอง หรือตัดหมวดนั้นออกจากเดโม